# 13 · The paper scout: a second agent, with someone else's MCP server

Stage 3, step 3. The OKF agent only knows the five papers in the bundle. The **paper scout** looks **outside**: it searches arXiv and suggests papers, including new ones.

Two new ideas in this step:
1. **Using a third-party MCP server.** We didn't write the arXiv tools. [`arxiv-mcp-server`](https://github.com/blazickjp/arxiv-mcp-server) is an open-source server; we just run it and connect, exactly the way we connect to our own OKF server. That's the point of MCP: any agent can use any server's tools.
2. **A conditional edge.** Until now every graph was a straight line. The scout **decides at run time** where to go next: if arXiv finds nothing, it loosens the search and tries again.

```
 "recent papers on mixture of experts"
        │
        ▼
     plan ──── qwen: request → arXiv search (JSON: keywords, newest first?, since when?)
        │
        ▼
     search ── MCP: search_papers on the arXiv server (:8001) ──► arxiv.org
        │
        ├── found papers ───────────────────────────┐
        └── nothing (1st try) ─► relax ─► search    │   (keywords OR-ed instead of AND-ed)
                                                    ▼
                                               summarize ── qwen: best ≤5 papers, one line each
                                                    │
                                                  check ── citations real? drop made-up papers, add links
                                                    │
                                                   log ── qa_log (agent = paper_scout)
```

Code: `okf_agent/scout.py` (the graph), `okf_agent/server.py` (serves it as a second agent), `langgraph.json` (a second entry).

## 0. Start the services

1. **Install** (once, `unsloth` env): `pip install -e ".[agent]"`. New: `uv`, which gives the `uvx` command.
2. **Restart the backend**: close the old windows, then run `start_backend.bat`. It now opens **three** windows:

| window | port | what |
|---|---|---|
| OKF MCP server | 8000 | our server (`search_okf`, …) |
| **arXiv MCP server** | **8001** | the third-party server, started with `uvx arxiv-mcp-server@0.8.2` |
| OKF agent server | 2024 | `langgraph dev`, now serving **two** agents: `okf_agent` and `paper_scout` |

**What `uvx` does:** it downloads the package into its **own private environment** (not `unsloth`), runs it, and caches it. So the arXiv server's dependencies can never clash with ours. `@0.8.2` pins the version, so an update can't silently change the output format our code reads. The first start takes about a minute (download); later starts are instant.

The arXiv server window should end with `Uvicorn running on http://127.0.0.1:8001`.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
import socket

import httpx
import pandas as pd
import psycopg

from okf_agent import AgentConfig
from okf_agent.a2a_client import A2AClient, langgraph_card_url
from okf_agent.mcp_tools import call_tool, load_tools

AGENT_URL = os.getenv("OKF_AGENT_URL", "http://127.0.0.1:2024")
config = AgentConfig.load()

def port_open(url):
    host, port = url.split("//")[1].split("/")[0].split(":")
    with socket.socket() as s:
        s.settimeout(2)
        return s.connect_ex((host, int(port))) == 0

print("OKF MCP server up:  ", port_open(config.mcp_url))
print("arXiv MCP server up:", port_open(config.arxiv_mcp_url), "  <- if False: see the arXiv window (first start downloads)")
print("agent server up:    ", port_open(AGENT_URL))
try:
    with psycopg.connect(config.postgres_url, connect_timeout=5):
        print("Postgres ok:         True")
except Exception as exc:
    print("Postgres FAILED:", exc)

OKF MCP server up:   True
arXiv MCP server up: True   <- if False: see the arXiv window (first start downloads)
agent server up:     True
Postgres ok:         True


## 1. Someone else's MCP server: what does it offer?

Connecting works exactly like with our own server: give `langchain-mcp-adapters` the URL, and it asks the server for its tool list (`tools/list`).

In [2]:
arxiv_tools = load_tools({"arxiv": {"url": config.arxiv_mcp_url, "transport": "streamable_http"}})
print(len(arxiv_tools), "tools:\n")
for name, tool in arxiv_tools.items():
    print(f"  {name:28s} {tool.description.splitlines()[0][:80]}")

19 tools:

  search_papers                Search arXiv by query with optional categories, date range, sort, and pagination
  download_paper               Download a paper from arXiv and return its text content. Tries the HTML version 
  list_papers                  List all papers that have been downloaded and stored locally via download_paper.
  read_paper                   Read the text content of a paper that was previously downloaded via download_pap
  get_abstract                 Fetch abstract and metadata by arXiv ID without downloading the paper. Use befor
  semantic_search              Semantic similarity search over papers you have already downloaded locally via d
  reindex                      Rebuild the local semantic index for downloaded papers.
  citation_graph               Return papers citing an arXiv paper and papers that it references using Semantic
  export_citations             Export BibTeX citations for one or more arXiv papers using authoritative arXiv m
  watc

It has 19 tools: search, abstracts, downloading and reading full papers, citation graphs, BibTeX, topic alerts…

The scout uses **only `search_papers`**. A small model like qwen 4B gets confused when it has to choose between 19 tools, so the graph decides which tool to call and qwen only writes text. Same idea as the OKF agent. The other tools are still there for later (the report agent could use `export_citations` for BibTeX).

The tool's **input schema**, written by the server's authors, says exactly what we may send:

In [3]:
schema = arxiv_tools["search_papers"].args_schema
props = schema["properties"] if isinstance(schema, dict) else schema.model_json_schema()["properties"]
for k, v in props.items():
    print(f"  {k:14s} {v.get('type', ''):8s} {v.get('description', '')[:90]}")

  query          string   arXiv query string. Prefer quoted phrases and ti:/au:/abs:/cat: field prefixes; AND/OR/AND
  max_results    integer  Maximum results to return (default: 5, max: 50).
  start          integer  Zero-based result offset (default: 0). Pass next_start from a previous response to fetch t
  abstract_mode  string   Abstract projection (default snippet ~280 chars, marked if truncated; full=complete; none=
  date_from      string   Inclusive start date (YYYY-MM-DD).
  date_to        string   Inclusive end date (YYYY-MM-DD).
  categories     array    arXiv category filters (e.g. ['cs.LG', 'cs.AI']). Strongly improves relevance.
  sort_by        string   Sort by 'relevance' (default) or 'date' (newest first).


## 2. Calling it by hand

One real arXiv search. Look at the raw result: plain JSON, with a `content_warning`. The server marks abstracts as **untrusted external content**. Anyone can write anything in an abstract, including "ignore your instructions…", so our summary prompt tells qwen to treat the results as data only. (Guardrails, stage 4, will go further.)

In [4]:
raw = call_tool(arxiv_tools["search_papers"], {"query": '"mixture of experts"', "max_results": 3,
                                               "sort_by": "date", "abstract_mode": "snippet"})
result = raw[0] if isinstance(raw, list) else raw
print("total matches on arXiv:", result.get("total_results"), "| returned:", result.get("returned"))
print("content_warning:", result.get("content_warning"), "\n")
for p in result.get("papers", []):
    print(f"[arXiv:{p['id']}] {p['title']} ({p['published'][:10]})")
    print("   ", p["abstract"][:200], "\n")

total matches on arXiv: 3914 | returned: 3

[arXiv:2610.22353] Rethinking Routing Exploration in Modern Mixture-of-Experts (2026-10-15)
    Mixture-of-experts (MoE) models have become an important approach for scaling model parameters. Modern MoE architectures increasingly combine two design choices: fine-grained expert segmentation and s 

[arXiv:2610.22344] MLExpert: Towards a Generalist Agentic Model for Autonomous Machine Learning Engineering (2026-10-15)
    Large language models (LLMs) are expanding from assisting scientific research and engineering to autonomously developing and improving AI systems, advancing AI for AI (AI4AI) as a pathway toward recur 

[arXiv:2610.22127] Explaining the Roles of Experts in Mixture-of-Experts LLMs for Personalization (2026-10-15)
    Personalization of Large Language Models (LLMs) is increasingly important for user-centric applications, yet prior methods mainly adapt dense LLMs, whose polysemantic neurons, each encoding multiple u 



## 3. The graph, node by node (`okf_agent/scout.py`)

| node | does | why this way |
|---|---|---|
| `plan` | qwen turns the request into JSON: `{"keywords": [...], "sort_by": "date", "date_from": "2024-01-01"}` | arXiv's search is keyword-based; "can you find me recent stuff on MoE?" works badly as a query. This call is tiny (~30 tokens out, about 1 s). If the JSON is broken, a plain rule takes the request's words instead, so a bad answer never stops the graph. |
| `search` | `search_papers` with the keywords **AND**-ed: every keyword must appear in the title or abstract | precise first |
| `relax` | only if nothing was found: keywords **OR**-ed, date filter dropped, search again | broader second try, **once** |
| `summarize` | qwen gets ≤6 papers (title, year, authors, abstract) and picks the best ≤5, one line each, citing `[arXiv:id]` | the one real generation |
| `check` | citations that aren't in the results are **made up**: those lines are **removed**. Then the **links are added by code**, never by qwen. | a 4B model sometimes invents plausible papers; a link written by code is always real |
| `log` | `qa_log` row, `agent = paper_scout` | same table as the OKF agent |

### The conditional edge

```python
g.add_edge("plan", "search")
g.add_conditional_edges("search", after_search, ["relax", "summarize"])   # ← decided at run time
g.add_edge("relax", "search")                                             # ← a loop back

def after_search(state):
    if state["passages"] or state.get("search_error"):
        return "summarize"
    if not state.get("relaxed") and (len(state["keywords"]) > 1 or "date_from" in state["arxiv_args"]):
        return "relax"                       # nothing found: one broader try
    return "summarize"                       # still nothing: summarize says "No relevant papers found."
```

`add_conditional_edges` takes a function that **looks at the state and returns the name of the next node**. The `relaxed` flag makes sure the loop runs at most once. Any loop in a graph needs a stop condition like this.

Drawn by LangGraph (dashed arrows = conditional):

In [5]:
from okf_agent.scout import build_scout_graph
print(build_scout_graph({}, None).compile().get_graph().draw_mermaid())
print("(paste into https://mermaid.live to see it drawn)")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan(plan)
	search(search)
	relax(relax)
	summarize(summarize)
	check(check)
	log(log)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan;
	check --> log;
	plan --> search;
	relax --> search;
	search -.-> relax;
	search -.-> summarize;
	summarize --> check;
	log --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

(paste into https://mermaid.live to see it drawn)


## 4. Watch one request go through the graph

This runs the scout **here in the notebook** (not through the server) so we can print what each node changed. It uses your real qwen and the real arXiv server.

In [6]:
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from okf.llm import make_llm

scout_local = build_scout_graph(arxiv_tools, make_llm(config.okf)).compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "nb13-demo"}}

for update in scout_local.stream({"messages": [HumanMessage("Find recent papers on direct preference optimization since 2024")]},
                                 cfg, stream_mode="updates"):
    for node, ch in update.items():
        ch = ch or {}
        if node == "plan":
            print(f"plan      keywords={ch['keywords']}  sort_by={ch['arxiv_args']['sort_by']}  "
                  f"date_from={ch['arxiv_args'].get('date_from')}  (from qwen: {ch['llm']['plan_from_llm']}, {ch['timings']['plan_ms']:.0f} ms)")
            print(f"          query sent: {ch['search_query']}")
        elif node == "search":
            print(f"search    {len(ch['passages'])} papers (arXiv total {ch.get('total_results')})  error={ch.get('search_error')}")
        elif node == "relax":
            print(f"relax     nothing found → retry with: {ch['search_query']}")
        elif node == "summarize":
            print(f"summarize qwen wrote {len(ch.get('answer', ''))} characters")
        elif node == "check":
            print(f"check     cited={[s['id'] for s in ch['sources']]}  made-up={ch['bad_citations']}")
        elif node == "log":
            print(f"log       total {ch['timings']['total_ms']:.0f} ms")

print("\n" + scout_local.get_state(cfg).values["messages"][-1].content)

plan      keywords=['direct preference optimization', 'reward modeling', 'preference learning']  sort_by=date  date_from=2024-01-01  (from qwen: True, 3259 ms)
          query sent: "direct preference optimization" AND "reward modeling" AND "preference learning"
search    6 papers (arXiv total 15)  error=None
summarize qwen wrote 1559 characters
check     cited=['arXiv:2609.32827', 'arXiv:2604.02766', 'arXiv:2602.06239', 'arXiv:2510.15716', 'arXiv:2510.18353']  made-up=[]
log       total 10674 ms

- Improving LLM Collaboration via Multi-Agent Preference Learning [arXiv:2609.32827] (2026): Presents multi-agent direct preference optimization (MADPO) as a component of a framework for enhancing LLM collaboration, directly addressing the request with recent work on direct preference optimization in multi-agent systems.  
- Random Is Hard to Beat: Active Selection in online DPO with Modern LLMs [arXiv:2604.02766] (2026): Evaluates direct preference optimization (DPO) in an online setting wit

Things to check in the output:
- **plan**: did qwen pick sensible keywords and `sort_by=date` (we said "recent") and `date_from=2024-01-01`?
- **search**: if it says 0 papers, the next line should be **relax** (the conditional edge at work).
- **check**: `made-up=[]` is what you want. If qwen invented a paper, it's listed here and its line is gone from the reply.
- **Links** at the bottom come from arXiv's ids, not from qwen.

## 5. Through the server, over A2A

`langgraph.json` now has two graphs, so `langgraph dev` serves **two A2A agents**, each with its own card and its own fixed id. Exactly the same client code talks to both.

In [7]:
okf = A2AClient.from_card(langgraph_card_url(AGENT_URL, "okf_agent"))
scout = A2AClient.from_card(langgraph_card_url(AGENT_URL, "paper_scout"))
for agent in (okf, scout):
    print(f"{agent.name:12s} {agent.url}\n             {agent.description}\n")

okf_agent    http://127.0.0.1:2024/a2a/bb1780c6-cc9f-5dd1-a605-b5608e80a356
             Answers questions about the OKF bundle (GPT-4 technical report and related papers) from retrieved passages, with [concept id] citations. Replies 'The bundle does not contain this.' when the answer isn't in the bundle.

paper_scout  http://127.0.0.1:2024/a2a/5211a61c-7534-573d-9c7a-962b40b98864
             Finds research papers on arXiv for a topic, including recent ones that are not in the OKF bundle. Returns up to 5 papers with [arXiv:id] citations, a one-line reason each, and links. Understands 'recent/latest' and 'since <year>'.



These two descriptions are what the **router** (step 14) will read to decide: *"What did GPT-4 score on the bar exam?"* → `okf_agent`; *"Any new papers on RLHF alternatives?"* → `paper_scout`.

In [8]:
r1 = scout.send("Find recent papers on mixture of experts language models")
print(r1.state, "\n" + r1.text)

TASK_STATE_COMPLETED 
- Horseshoe Mixtures-of-Experts (HS-MoE) [arXiv:2601.09043] (2026): Presents a Bayesian MoE framework with data-adaptive sparsity and connects to modern LLMs under extreme sparsity constraints.  
- EAC-MoE: Expert-Selection Aware Compressor for Mixture-of-Experts Large Language Models [arXiv:2508.01625] (2025): Addresses GPU memory and inference latency in MoE-LLMs via quantization calibration and expert pruning aligned with selection dynamics.  
- Multi-level context Modeling for consistent expert selection in Mixture-of-Experts [arXiv:2607.16427] (2026): Improves expert routing consistency and performance by integrating cross-layer and local context signals in MoE models.  
- XShare: Collaborative in-Batch Expert Sharing for Faster MoE Inference [arXiv:2602.07265] (2026): Dynamically reduces expert activation and GPU load in batched inference through batch-aware, greedy expert selection.  
- MoE-GRPO: Optimizing Mixture-of-Experts via Reinforcement Learning in V

In [9]:
r2 = scout.send("only ones since 2025", context_id=r1.context_id)     # follow-up in the same conversation
print(r2.state, "| same conversation:", r2.context_id == r1.context_id, "\n" + r2.text)

TASK_STATE_COMPLETED | same conversation: True 
All results are from 2026, which is since 2025.

- Graph neural networks and the energetic cavity method for combinatorial optimization [arXiv:2609.07456] (2026): Explores modified graph neural networks incorporating mean-field heuristics for solving Ising models, relevant to combinatorial optimization and deep learning performance.  
- "What's That Sound?": A Versatile, Robust, and Lightweight Convolutional Transformer for Environment Sound Recognition [arXiv:2609.27762] (2026): Presents a lightweight deep learning model for real-time sound recognition, suitable for mobile deployment and safety applications.  
- Quantum-Inspired Robust and Scalable SAR Object Classification [arXiv:2604.25755] (2026): Investigates tensor networks for robust SAR image classification, addressing noise and edge-device deployment challenges.  
- HoloPASWIN: Robust Inline Holographic Reconstruction via Physics-Aware Swin Transformers [arXiv:2603.04926] (2026):

The follow-up "only ones since 2025" is short, so `plan` treats it as a follow-up: qwen gets the earlier request too, and should keep the topic and add `date_from`.

## 6. Both agents in one log

`qa_log` now has rows from both agents. For the scout, `search_query` is the arXiv query, `cited` the arXiv ids it recommended, and `passages` every paper it was shown.

In [10]:
with psycopg.connect(config.postgres_url) as conn:
    cur = conn.execute("""SELECT id, agent, followup, search_query, cited, bad_citations, refused, round(total_ms) AS ms
                          FROM qa_log ORDER BY id DESC LIMIT 6""")
    recent = pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
    cur = conn.execute("""SELECT agent, count(*) AS questions, round(avg(total_ms)) AS avg_ms,
                                 sum(CASE WHEN jsonb_array_length(bad_citations) > 0 THEN 1 ELSE 0 END) AS with_made_up_citations
                          FROM qa_log GROUP BY agent""")
    per_agent = pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
display(per_agent)
recent

,agent,questions,avg_ms,with_made_up_citations
0,okf_agent,15,9470.0,1
1,paper_scout,2,9612.0,0


,id,agent,followup,search_query,cited,bad_citations,refused,ms
0,17,paper_scout,False,"""deep learning"" AND ""neural networks"" AND ""mod...","[arXiv:2609.07456, arXiv:2609.27762, arXiv:260...",[],False,9601.0
1,16,paper_scout,False,"""mixture of experts"" AND ""language model"" AND ...","[arXiv:2601.09043, arXiv:2508.01625, arXiv:260...",[],False,9623.0
2,15,okf_agent,False,What is RLHF?,[topics/rlhf],[],False,8529.0
3,14,okf_agent,True,What score did GPT-4 get on the bar exam? And ...,"[topics/simulated-exams, papers/gpt-4-technica...",[],False,6312.0
4,13,okf_agent,False,What score did GPT-4 get on the bar exam?,"[topics/simulated-exams, papers/gpt-4-technica...",[],False,5734.0
5,12,okf_agent,False,What score did GPT-4 get on the bar exam?,"[topics/simulated-exams, papers/gpt-4-technica...",[],False,13204.0


## 7. When arXiv is slow or says no

arXiv allows about one request every 3 seconds. The arXiv server already waits and retries. If it still fails (rate limit, no internet), `search_papers` returns an error **status** instead of papers. The scout then answers *"I couldn't search arXiv right now (…)"* with `refused = True` in `qa_log`. The task still completes; the router can pass that message on.

If the **arXiv MCP server window** is closed, the A2A task comes back `FAILED`, like the OKF agent without its MCP server in `12`.

## Where we are

- ✅ **10** okf_mcp · ✅ **11** okf_agent · ✅ **12** A2A service
- ✅ **13** **paper_scout**: third-party arXiv MCP server (via `uvx`), conditional edge with a retry, made-up papers removed, links by code
- ⏭ **14** **router**: one entry point. It reads both agent cards, decides per question (qwen with a fixed set of choices), and forwards over A2A. It keeps the chat; the agents just answer.
- ⏭ **15** report agent (filesystem MCP server)